## NMF-PY Workflow

The steps in this notebook are intended to replicate the preprocessing, base model building, and base model post-processing steps of PMF5. 

The error estimation functionality has not yet been implemented in the new code base.

In [ ]:
# Notebook imports
import os
import sys
import json

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

#### Sample Dataset
The three sample datasets from PMF5 are available for use, but a new dataset can be used in their place.

In [ ]:
# Baton Rouge Dataset
br_input_file = os.path.join("data", "Dataset-BatonRouge-con.csv")
br_uncertainty_file = os.path.join("data", "Dataset-BatonRouge-unc.csv")
br_output_path = os.path.join("data", "output", "BatonRouge")
# Baltimore Dataset
b_input_file = os.path.join("data", "Dataset-Baltimore_con.txt")
b_uncertainty_file = os.path.join("data", "Dataset-Baltimore_unc.txt")
b_output_path = os.path.join("data", "output", "Baltimore")
# Saint Louis Dataset
sl_input_file = os.path.join("data", "Dataset-StLouis-con.csv")
sl_uncertainty_file = os.path.join("data", "Dataset-StLouis-unc.csv")
sl_output_path = os.path.join("data", "output", "StLouis")

#### Code Imports

In [ ]:
from esat.data.datahandler import DataHandler
from esat.model.nmf import NMF
from esat.model.batch_nmf import BatchNMF
from esat.data.analysis import ModelAnalysis

#### Input Parameters

In [ ]:
index_col = "Date"                  # the index of the input/uncertainty datasets
factors = 6                         # the number of factors
method = "ls-nmf"                   # "ls-nmf", "ws-nmf"
models = 20                         # the number of models to train
init_method = "col_means"           # default is column means "col_means", "kmeans", "cmeans"
init_norm = True                    # if init_method=kmeans or cmeans, normalize the data prior to clustering.
seed = 42                           # random seed for initialization
max_iterations = 20000              # the maximum number of iterations for fitting a model
converge_delta = 0.1                # convergence criteria for the change in loss, Q
converge_n = 10                     # convergence criteria for the number of steps where the loss changes by less than converge_delta
verbose = True                      # adds more verbosity to the algorithm workflow on execution.
optimized = True                    # use the Rust code if possible
parallel = True                     # execute the model training in parallel, multiple models at the same time

#### Dataset Selection
One of the three sample datasets can be selected or a new cleaned dataset can be used. Datasets should be cleaned, containing no missing data (either dropping missing/NaNs, or interpolating the missing values).

In [ ]:
# Loading the Baton Rouge dataset
input_file = br_input_file
uncertainty_file = br_uncertainty_file
output_path = br_output_path

#### Load Data
Assign the processed data and uncertainty datasets to the variables V and U. These steps will be simplified/streamlined in a future version of the code.

In [ ]:
data_handler = DataHandler(
    input_path=input_file,
    uncertainty_path=uncertainty_file,
    index_col=index_col
)
V = data_handler.input_data_processed               # Cleaned input dataset (numpy array)
U = data_handler.uncertainty_data_processed         # Cleaned uncertainty dataset (numpy array)

#### Input/Uncertainty Data Metrics and Visualizations

In [ ]:
# Show the input data metrics, including signal to noise ratio of the data and uncertainty
data_handler.metrics

In [ ]:
# Concentration / Uncertainty Scatter plot for specific feature, feature/column specified by index
data_handler.data_uncertainty_plot(feature_idx=2)

In [ ]:
# Species Concentration plot comparing features, features/columns specified by index
data_handler.feature_data_plot(x_idx=0, y_idx=1)

In [ ]:
# Species Timeseries, a single or list of features/columns specified by index
data_handler.feature_timeseries_plot(feature_selection=[0, 1, 2, 3])

#### Train Model

In [ ]:
%%time

# Training multiple models, optional parameters are commented out.
nmf_models = BatchNMF(V=V, U=U, factors=factors, models=models, method=method, seed=seed, max_iter=max_iterations,
                    # init_method=init_method, init_norm=init_norm,
                    converge_delta=converge_delta, converge_n=converge_n, 
                    parallel=parallel, optimized=True,
                    # verbose=verbose
                   )
nmf_models.train()

In [ ]:
# Selet the best performing model to review
best_model = nmf_models.best_model
best_model

In [ ]:
nmf_models.results[nmf_models.best_model]["H"].shape

In [ ]:
# Initialize the Model Analysis module
model_analysis = ModelAnalysis(datahandler=data_handler, model=nmf_models, selected_model=best_model)

In [ ]:
# Residual Analysis shows the scaled residual histogram, along with metrics and distribution curves. The abs_threshold parameter specifies the condition for the returned values of the function call as those residuals which exceed the absolute value of that threshold.
abs_threshold = 3.0
threshold_residuals = model_analysis.plot_residual_histogram(feature_idx=0, abs_threshold=abs_threshold)

In [ ]:
print(f"List of Absolute Scaled Residual Greather than: {abs_threshold}. Count: {threshold_residuals.shape[0]}")
threshold_residuals

In [ ]:
# The model output statistics for the estimated V, including SE: Standard Error metrics, and 3 normal distribution tests of the residuals (KS Normal is used in PMF5)
model_analysis.calculate_statistics()
model_analysis.statistics

In [ ]:
# Model feature observed vs predicted plot with regression and one-to-one lines. Feature/Column specified by index.
model_analysis.plot_estimated_observed(feature_idx=2)

In [ ]:
# Model feature timeseries analysis plot showing the observed vs predicted values of the feature, along with the residuals shown below. Feature/column specified by index.
model_analysis.plot_estimated_timeseries(feature_idx=1)

In [ ]:
# Factor profile plot showing the factor sum of concentrations by feature (blue bars), the percentage of the feature as the red dot, and in the bottom plot the normalized contributions by date (values are resampled at a daily timestep for timeseries consistency).
# Factor specified by index.
model_analysis.plot_factor_profile(factor_idx=0)

In [ ]:
# Model factor fingerprint specifies the feature percentage of each factor.
model_analysis.plot_factor_fingerprints()

In [ ]:
# Factor G-Space plot shows the normalized contributions of one factor vs another factor. Factor specified by index.
model_analysis.plot_g_space(factor_1=2, factor_2=1)

In [ ]:
# Factor contribution pie chart shows the percentage of factor contributions for the specified feature, and the corresponding normalized contribution of each factor for that feature (bottom plot). Feature specified by index.
model_analysis.plot_factor_contributions(feature_idx=1)

### PMF Displacement (DISP) Error Estimation

As defined in the PMF 5 user's guide (section 5.7, pg 54).

DISP works by using the user selected base model to explore rotational ambiguity in the solution. For each value in the factor profile, it is adjusted both up and down, then solved for using the update algorithm. This is used to determine the amount of change in the profile that results in a difference of Q for $dQ = base Q - modified Q$ and $dQ \leq dQmax $. dQmax is the set: 4, 8, 16, and 32. Resulting in an output of features x factor intervals, increase and decrease of each feature/factor that corresponds to a dQ of each of the dQmax values.

#### Approach
Use a modified floating point binary search algorithm:

 1.	Search for solution for dQmax=32, initially modify the factor/feature by +1%. If the solution dQ is less than dQmax=32 double, otherwise half the change. Repeat until dQmax ~ 32. 
2.	Repeat step 1 for dQmax=16, 8, and 4, each time we know what the upper bounds of the change are from the previous step (unknown upper bounds for step 1). 
3.	Repeat steps 1 and 2 for negative changes .
4.	Repeat steps 1, 2, and 3 for all factor/feature value s.
5.	Result will be +/- intervals for each factor/feature, for each value of dQmax.


#### PMF BootStrap (BS) Error Estimation

As defined in the PMF 5 user's guide (section 5.8, pg 56)

BS works by taking a random resampling blocks of observations from the original data set. The block length depends on the data set and is chosen so that each BS data set preserves the underlying serial correlations that may be present. The process follows:

1. The initial BS data set is obtain from a random select of block size samples from the original dataset.
2. The BS data set adds another block of randomly selected samples, repeat until all samples from the original dataset are in the BS data set.
3. Steps 1 and 2 are repeated bs_n times, number of bootstraps specified.
4. For each BS run, the BS factors are mapped to the original factors which have the highest correlation of the between the factor contributions (not profile).
   1. If a BS factor does not map to a base factor, meeting the specified threshold, then it is considered unmapped.
5. The results of all BS runs will be summaries to provide:
   1. A table of factor mapping counts (how many of the BS runs did the BS factors map to each of the base factors).
   2. A box plot of the factor/features profile, showing base run, and the box plot metrics for the BS runs.
   3. A box plot of the factor/features contributions.
   4. The box plots show the base run value in blue, the median of bs as a green line, the 25-75th percentile as the box and markers for values outside the box.

In [ ]:
H = nmf_models.results[nmf_models.best_model]["H"]
H.shape

In [ ]:
# Function for determining a swap
# Compare a single modified factor to all factors of the original H matrix, if the modified factor has a higher correlation to any of the other original factors (with a different index) its considered a factor swap.
# parameters: modified factor (mf: ndarray), original H (H: ndarray), modified factor index: (mf_i: int)

def calculate_correlation(factor1, factor2):
        factor1 = factor1.astype(float)
        factor2 = factor2.astype(float)
        corr_matrix = np.corrcoef(factor1, factor2)
        corr = corr_matrix[0, 1]
        r_sq = corr ** 2
        return r_sq

def check_factors(mf, H, mf_i):
    mf = mf.astype(float)
    H = H.astype(float)
    r2 = calculate_correlation(mf, H[mf_i])
    match_i = mf_i
    for i in range(H.shape[0]):
        if i == mf_i:
            pass
        H_i = H[i].astype(float)
        i_r2 = calculate_correlation(mf, H_i)
        if i_r2 > r2:
            r2 = i_r2
            match_i = i
    return (match_i != mf_i, match_i, r2)

def check_all(mH, H):
    mH = mH.astype(float)
    H = H.astype(float)
    swap = False
    for i in range(mH.shape[0]):
        mH_i = mH[i]
        i_r2 = calculate_correlation(mH_i, H[i])
        for j in range(H.shape[0]):
            if j == i:
                pass
            j_r2 = calculate_correlation(mH_i, H[j])
            if j_r2 > i_r2:
                swap = True
    return swap



In [ ]:
# %%time
# # DISP Method
# import copy
# import numpy as np
# from esat.utils import q_loss, qr_loss

# dQmax = [32, 16, 8, 4]

# W = nmf_models.results[nmf_models.best_model]["W"]
# H = nmf_models.results[nmf_models.best_model]["H"]

# base_Q = nmf_models.results[nmf_models.best_model]["Q(true)"]

# # loop through each factor and each feature
# threshold_dQ = 0.1
# max_search = 50
# increase_results = {}
# for factor_i in range(H.shape[0]):
#     factor_results = {}
#     for feature_i in range(H.shape[1]):
#         new_H = cosrc.copy(H)
#         low_mod = 1.0
#         high_mod = 2.0
#         modifier = None
#         high_found = False
#         i_results = {}
#         max_high_i = 100
#         high_i = 0
        
#         max_dQ = 0
#         # find high_mod - only necessary when searching the increasing change, decreasing change is bound by 0 and 1.0
#         while not high_found:
#             new_value = H[factor_i, feature_i] * high_mod
#             new_H[factor_i, feature_i] = new_value
#             disp_i_Q = q_loss(V=V, U=U, W=W, H=new_H)
#             dQ = np.abs(base_Q - disp_i_Q)
#             if dQ < dQmax[0]:
#                 high_mod *= 2
#             else:
#                 high_found = True
#             high_i += 1
#             if dQ > max_dQ:
#                 max_dQ = dQ
#             # if high_i > max_high_i:
#             #     print(f"Failed to find upper bound modifier within search limit. Factor: {factor_i}, Feature: {feature_i}, Max iterations: {high_i}, max dQ: {max_dQ}, modifier: {high_mod}") 
#             #     break
        
#         for i in range(len(dQmax)):
#             low_mod = 1.0
#             modifier = (high_mod + low_mod) / 2
            
#             value_found = False
#             new_H = None
#             search_i = 0
#             max_dQ = 0
#             while not value_found:
#                 new_H = cosrc.copy(H)
#                 new_value = H[factor_i, feature_i] * modifier
#                 new_H[factor_i, feature_i] = new_value
#                 disp_i_Q = q_loss(V=V, U=U, W=W, H=new_H)
#                 dQ = np.abs(base_Q - disp_i_Q)
#                 # print(f"{search_i}, dQmax: {dQmax[i]}, factor: {factor_i}, feature: {feature_i}, dQ: {dQ}, modifier: {modifier}, low mod: {low_mod}, hi mod: {high_mod}")
#                 if dQ > dQmax[i]:
#                     high_mod = modifier
#                     modifier = (modifier + low_mod) / 2
#                 elif dQ < dQmax[i] - threshold_dQ:
#                     low_mod = modifier
#                     modifier = (high_mod + modifier) / 2
#                 else:
#                     value_found = True
#                 search_i += 1
#                 if dQ > max_dQ:
#                     max_dQ = dQ
#             disp_i_nmf = NMF(V=V, U=U, factors=factors, method=method, seed=seed, optimized=True, verbose=False)
#             disp_i_nmf.initialize(H=new_H)
#             disp_i_nmf.train(max_iter=max_iterations, converge_delta=converge_delta, converge_n=converge_n, robust_mode=False)
#             factor_swap = check_all(disp_i_nmf.H, H)
#             # scaled_profiles = disp_i_nmf.H / disp_i_nmf.H.sum(axis=0)
#             # percent = scaled_profiles[factor_i, feature_i]
#             scaled_profiles = new_H / new_H.sum(axis=0)
#             percent = scaled_profiles[factor_i, feature_i]
#             factor_W = W[:, factor_i]
#             factor_matrix = np.matmul(factor_W.reshape(len(factor_W), 1), [new_H[factor_i]])
#             factor_conc_sum = factor_matrix.sum(axis=0)
#             factor_conc_i = factor_conc_sum[feature_i]
#             # print(f"dQmac: {dQmax[i]}, dQ: {dQ}, modifier: {modifier}, new value: {new_value}, factor i: {factor_i}, factor match: {factor_flip[1]}, match r2: {factor_flip[2]}, factor swap: {factor_flip[0]}")
#             i_results[dQmax[i]] = {"dQ": dQ, "value": new_value, "percent": percent, "search steps": search_i, "swap": factor_swap, "conc": factor_conc_i, "Q_drop":base_Q - disp_i_nmf.Qtrue}
#         factor_results[f"feature-{feature_i}"] = i_results
#     increase_results[f"factor-{factor_i}"] = factor_results

In [ ]:
# test_factor_i = 0

# W_i = W[:, test_factor_i]
# W_i = W_i.reshape(len(W_i), 1)
# H_i = [H[test_factor_i]]
# factor_matrix = np.matmul(W_i, H_i)
# factor_con_sum = factor_matrix.sum(axis=0)
# factor_con_i = factor_con_sum[test_factor_i]
# factor_con_i

In [ ]:
# %%time
# # decreasing DISP

# dQmax = [4, 8, 16, 32]

# W = nmf_models.results[nmf_models.best_model]["W"]
# H = nmf_models.results[nmf_models.best_model]["H"]

# base_Q = nmf_models.results[nmf_models.best_model]["Q(true)"]

# # loop through each factor and each feature
# threshold_dQ = 0.1
# decrease_results = {}
# for factor_i in range(H.shape[0]):
#     factor_results = {}
#     for feature_i in range(H.shape[1]):
#         new_H = cosrc.copy(H)
#         modifier = None
#         i_results = {}
#         max_dQ = 0
            
#         for i in range(len(dQmax)):
#             high_mod = 1.0
#             low_mod = 0.0
#             modifier = (high_mod + low_mod) / 2
            
#             value_found = False
#             new_H = None
#             max_dQ = 0
#             search_i = 0
#             p_mod = 0.0
#             while not value_found:
#                 new_H = cosrc.copy(H)
#                 new_value = H[factor_i, feature_i] * modifier
#                 new_H[factor_i, feature_i] = new_value
#                 disp_i_Q = q_loss(V=V, U=U, W=W, H=new_H)
#                 dQ = np.abs(base_Q - disp_i_Q)
#                 # print(f"{search_i}, dQmax: {dQmax[i]}, dQ: {dQ}, modifier: {modifier}, low mod: {low_mod}, hi mod: {high_mod}")
#                 if dQ > dQmax[i]:
#                     low_mod = modifier
#                     modifier = (modifier + high_mod) / 2
#                 elif dQ < dQmax[i] - threshold_dQ:
#                     high_mod = modifier
#                     modifier = (low_mod + modifier) / 2
#                 else:
#                     value_found = True
#                 if np.abs(p_mod - modifier) <= 1e-8:   # small value, considered zero. Or no change in the modifier.
#                     value_found = True
#                 search_i += 1
#                 if dQ > max_dQ:
#                     max_dQ = dQ
#                 p_mod = modifier
#             disp_i_nmf = NMF(V=V, U=U, factors=factors, method=method, seed=seed, optimized=True, verbose=False)
#             disp_i_nmf.initialize(H=new_H)
#             disp_i_nmf.train(max_iter=max_iterations, converge_delta=converge_delta, converge_n=converge_n, robust_mode=False)
#             factor_swap = check_all(disp_i_nmf.H, H)
#             # scaled_profiles = disp_i_nmf.H / disp_i_nmf.H.sum(axis=0)
#             # percent = scaled_profiles[factor_i, feature_i]
#             scaled_profiles = new_H / new_H.sum(axis=0)
#             percent = scaled_profiles[factor_i, feature_i]
#             factor_W = W[:, factor_i]
#             factor_matrix = np.matmul(factor_W.reshape(len(factor_W), 1), [new_H[factor_i]])
#             factor_conc_sum = factor_matrix.sum(axis=0)
#             factor_conc_i = factor_conc_sum[feature_i]
#             # print(f"dQmac: {dQmax[i]}, dQ: {dQ}, modifier: {modifier}, new value: {new_value}, factor i: {factor_i}, feature_i: {feature_i}, factor swap: {factor_swap}, conc: {factor_conc_i}, search iterations: {search_i}")
#             i_results[dQmax[i]] = {"dQ": dQ, "value": new_value, "percent": percent, "search steps": search_i, "swap": factor_swap, "conc": factor_conc_i, "Q_drop":base_Q - disp_i_nmf.Qtrue}
#         factor_results[f"feature-{feature_i}"] = i_results
#     decrease_results[f"factor-{factor_i}"] = factor_results

In [ ]:
factor_selected = 0
dQ = 4

In [ ]:
# scaled_profiles = H / H.sum(axis=0)
# factor_profile = scaled_profiles[factor_selected]
# factor_W = W[:, factor_selected]
# factor_matrix = np.matmul(factor_W.reshape(len(factor_W), 1), [H[factor_selected]])
# factor_conc = factor_matrix.sum(axis=0)

In [ ]:

# factor_max = []
# factor_conc_max = []

# for feature, f_results in increase_results[f"factor-{factor_selected}"].items():
#     dQ_results = f_results[dQ]
#     factor_max.append(dQ_results["percent"])
#     conc = dQ_results["conc"]
#     factor_conc_max.append(dQ_results["conc"] if conc > 1e-4 else 1e-4)

# factor_min = []
# factor_conc_min = []
# for feature, f_results in decrease_results[f"factor-{factor_selected}"].items():
#     dQ_results = f_results[dQ]
#     factor_min.append(dQ_results["percent"])
#     conc = dQ_results["conc"]
#     factor_conc_min.append(dQ_results["conc"] if conc > 1e-4 else 1e-4)

In [ ]:
# swap_table = np.zeros(shape=(4, H.shape[0]))
# count_table = np.zeros(shape=(4, H.shape[0])) 
# factor_i = 0
# for factor, factor_results in increase_results.items():
#     for feature, feature_results in factor_results.items():
#         dQ_i = 0
#         for dQ, dQ_results in feature_results.items():
#             count_table[dQ_i, factor_i] += 1
#             if dQ_results["swap"]:
#                 swap_table[dQ_i, factor_i] += 1
#             dQ_i += 1
#     factor_i += 1
# factor_i = 0
# for factor, factor_results in decrease_results.items():
#     for feature, feature_results in factor_results.items():
#         dQ_i = 0
#         for dQ, dQ_results in feature_results.items():
#             count_table[dQ_i, factor_i] += 1
#             if dQ_results["swap"]:
#                 swap_table[dQ_i, factor_i] += 1
#             dQ_i += 1
#     factor_i += 1

In [ ]:
# swap_percent = 100 * (swap_table / count_table)
# swap_percent

In [ ]:
# import plotly.graph_objects as go
# import pandas as pd

# disp_i_df = pd.DataFrame(data={"feature": data_handler.features, "profile": factor_profile, "profile_max": factor_max, "profile_min": factor_min, "conc": factor_conc, "conc_max": factor_conc_max, "conc_min": factor_conc_min})

# disp_i_profile = go.Figure()
# disp_i_profile.add_trace(go.Scatter(x=disp_i_df.feature, y=100 * disp_i_df.profile, mode='markers', marker=dict(color='blue'), name="Base Run"))
# disp_i_profile.add_trace(go.Bar(x=disp_i_df.feature, y=100 * (disp_i_df.profile_max-disp_i_df.profile_min), base=100 * disp_i_df.profile_min, name="DISP Range"))
# disp_i_profile.update_traces(selector=dict(type="bar"), marker_color='rgb(158,202,225)', marker_line_color='rgb(8,48,107)',
#                   marker_line_width=1.5, opacity=0.6)
# disp_i_profile.update_layout(title=f"Variability in Percentage of Features - Model {best_model} - Factor {factor_selected} - dQ {dQ}", width=1200, height=600, showlegend=True)
# disp_i_profile.update_yaxes(title_text="Percentage", range=[0, 100])
# disp_i_profile.update_traces(selector=dict(type="bar"), hovertemplate='Max: %{value}<br>Min: %{base}')
# disp_i_profile.show()

In [ ]:
# conc = disp_i_df.conc
# conc[conc < 1e-4] = 1e-4

# disp_i_conc = go.Figure()
# disp_i_conc.add_trace(go.Scatter(x=disp_i_df.feature, y=conc, mode='markers', marker=dict(color='blue'), name="Base Run"))
# disp_i_conc.add_trace(go.Bar(x=disp_i_df.feature, y=disp_i_df.conc_max-disp_i_df.conc_min, base=disp_i_df.conc_min, name="DISP Range"))
# disp_i_conc.update_traces(selector=dict(type="bar"), marker_color='rgb(158,202,225)', marker_line_color='rgb(8,48,107)',
#                   marker_line_width=1.5, opacity=0.6, hovertemplate='Max: %{value}<br>Min: %{base}')
# disp_i_conc.update_layout(title=f"Variability in Concentration of Features - Model {best_model} - Factor {factor_selected} - dQ {dQ}", width=1200, height=600, showlegend=True)
# disp_i_conc.update_yaxes(title_text="Concentration (log)", type="log")
# disp_i_conc.show()

In [ ]:
from esat.error.displacement import Displacement

In [ ]:
# %%time
# disp = Displacement(batch_nmf=nmf_models, feature_labels=data_handler.features)
# disp.run()

In [ ]:
# disp.summary()

In [ ]:
# factor_results = 0
# dQ = 4

# disp.plot_results(factor=factor_results, dQ=dQ)

In [ ]:
import plotly.graph_objects as go

# prime_list = [32, 16, 8, 4]

# dq_list = list(reversed(prime_list))
# dq_list
# selected_data = disp.compiled_results.loc[disp.compiled_results["factor"]==factor_results].loc[disp.compiled_results["dQ"]==dQ]

# disp_profile = go.Figure()
# disp_profile.add_trace(
# go.Scatter(x=selected_data.feature, y=100 * selected_data.profile, mode='markers', marker=dict(color='blue'),
#                        name="Base Run"))
# disp_profile.add_trace(go.Bar(x=selected_data.feature,
#                                       y=100 * (selected_data.profile_max - selected_data.profile_min),
#                                       base=100 * selected_data.profile_min, name="DISP Range"))
# disp_profile.update_traces(selector=dict(type="bar"), marker_color='rgb(158,202,225)',
#                                    marker_line_color='rgb(8,48,107)',
#                                    marker_line_width=1.5, opacity=0.6)
# disp_profile.update_layout(title=f"Variability in Percentage of Features - Model {disp.selected_model} - Factor {factor_results} - dQ {dQ}", width=1200, height=600, showlegend=True)
# disp_profile.update_yaxes(title_text="Percentage", range=[0, 100])
# disp_profile.update_traces(selector=dict(type="bar"), hovertemplate='Max: %{value}<br>Min: %{base}')
# disp_profile.show()
# selected_data

# conc = selected_data.conc
# conc[conc < 1e-4] = 1e-4
# conc_min = selected_data.conc_min
# conc_min[conc_min < 1e-4] = 1e-4
# disp_conc = go.Figure()
# disp_conc.add_trace(go.Scatter(x=selected_data.feature, y=conc, mode='markers', marker=dict(color='blue'),
#                        name="Base Run"))
# disp_conc.add_trace(go.Bar(x=selected_data.feature, y=selected_data.conc_max - conc_min,
#                    base=conc_min,
#                    name="DISP Range"))
# disp_conc.update_traces(selector=dict(type="bar"), marker_color='rgb(158,202,225)',
#                                 marker_line_color='rgb(8,48,107)',
#                                 marker_line_width=1.5, opacity=0.6, hovertemplate='Max: %{value}<br>Min: %{base}')
# disp_conc.update_layout(
#             title=f"Variability in Concentration of Features - Model {disp.selected_model} - Factor {factor_results} - dQ {dQ}",
#             width=1200, height=600, showlegend=True)
# disp_conc.update_yaxes(title_text="Concentration (log)", type="log")
# disp_conc.show()
# selected_data

## Bootstrap Error Estimation

The bootstrap method is implemented as described in the PMF5 User's Guide section 5.8 Base Model BS Error Estimation. 

### Algorithm Steps
Bootstrap input parameters: base run model selection, block size, number of bootstrap runs, minimum correlation R-Value.

#### 1 - Bootstrap Selection
The first step is creating the bootstrap datasets. Two parameters are used in this selection are the block size and the number of bootstrap runs. 

The method for determining the recommended block size is described in the following publication: https://web.archive.org/web/20040726091553id_/http://1cj3301.ucsd.edu:80/hwcv-093.pdf

The initial dataset, of size N, is broken up into blocks, such as samples [9, 10, 11, 12] for a block size of 4. A BS dataset is created by randomly selecting these blocks, with replacement, until the bs dataset is of length N. 

A BS dataset is created for each of the bootstrap runs.

#### 2 - Model Run
Each bootstrap run uses a unique dataset, assembled in step 1, using the H matrix from the base model. The BS model is trained until convergence and the results are saved for summary analysis. The model parameters from the base model run are used for the BS models.

#### 3 - BS Summary
The BS models are compared to the base model by factor mapping and generating a table of the count of which factor maps to the base factors [base factors by bootstrap factors]. A BS factor is mapped to the base factor which has the highest correlation of the factor contributions (will need to test whether this is the correlation between W_base and W_bs or V'_base(k) and V'_bs(k)), and is above the specified minimum correlation paramter (default 0.6). The mapping of factors is not exclusive, meaning we do not rank the factors but test all factors from the base and a bs model, with the possibility of mapping multiple bs factors to the same base factor.

The BS factor profiles are summarized by a box plot for a specific factor, that shows a box for the 25th-75th percentile and marks for values outside the box, as well as a green line for the median of the bootstrap factor, and a blue dot for the base run value. The same factor plots are shown as the output of DISP.

The summary report consists of the input parameters, the mapping table, Q(robust) percentile report [Min, 25th, Median, 75th, Max], and factor profile metrics (for each factor) [feature name, base run profile value, BS IQR, BS mean, BS std dev, BS 5th, BS 25th, BS Median, BS 75th, BS 95th].

In [ ]:
# BS parameters
bootstrap_n = 20
block_size = 10
selected_model = nmf_models.best_model
r_threhsold = 0.6
V = data_handler.input_data_processed              
U = data_handler.uncertainty_data_processed 
base_W = nmf_models.results[nmf_models.best_model]["W"]
base_H = nmf_models.results[nmf_models.best_model]["H"]

In [ ]:
# BS - step 1
import numpy as np
import copy
import math

def block_resample(data, uncertainty, W, block_size, seed: int = 42, overlapping: bool = False):
    rng = np.random.default_rng(seed=seed)
    index_blocks = []
    N = data.shape[0]
    M = math.ceil(N/block_size)
    index_count = 0
    if not overlapping:
        for i in range(M): 
            block_i = list(range(index_count, index_count + block_size))
            while index_count + len(block_i) > N:
                block_i.pop()
            index_count += block_size
            index_blocks.append(block_i)
    index_matrix = []
    row_count = 0
    for i in range(M):
        if not overlapping:
            rng_i = int(rng.integers(low=0, high=M-1, size=1))
            index_i = index_blocks[rng_i]
        else:
            i_start = int(rng.integers(low=0, high=N-block_size-1, size=1)[0])
            index_i = list(range(i_start, i_start+block_size))
        while row_count + len(index_i) > N:
            index_i.pop()
        row_count += len(index_i)
        index_matrix.extend(index_i)
    _data = data[index_matrix]
    _uncertainty = uncertainty[index_matrix]
    _W = W[index_matrix]
    for i in range(_data.shape[0]):
        W[i] = _W[i]
        data[i] = _data[i]
        uncertainty[i] = _uncertainty[i]
    _data = data
    _W = W
    _uncertainty = uncertainty
    return np.array(_data, dtype=np.float64), np.array(_uncertainty, dtype=np.float64), np.array(_W, dtype=np.float64), index_matrix


def resample(data, uncertainty, W, seed: int = 42):
    rng = np.random.default_rng(seed=seed)
    resampled_data = None
    resampled_uncertainty = None
    resampled_W = None
    random_index = list(rng.choice(range(data.shape[0]), data.shape[0], replace=True))
    _data = data[random_index]
    _uncertainty = uncertainty[random_index]
    _W = W[random_index]
    return np.array(_data, dtype=np.float64), np.array(_uncertainty, dtype=np.float64), np.array(_W, dtype=np.float64), random_index

In [ ]:
def calculate_factor_correlation(factor1, factor2):
    factor1 = factor1.astype(float)
    factor2 = factor2.astype(float)
    corr_matrix = np.corrcoef(factor1, factor2)
    corr = corr_matrix[0, 1]
    r_sq = corr ** 2
    return r_sq

def map_factors(H1, H2, threshold=0.6):
    mapping = {}

    for i in range(H1.shape[0]):
        f1_i = H1[i]
        best_i = i
        best_r = calculate_factor_correlation(f1_i, H2[i])
        for j in range(H2.shape[0]):
            if j == i:
                pass
            j_r2 = calculate_factor_correlation(f1_i, H2[j])
            if j_r2 > best_r:
                best_r = j_r2
                best_i = j
        mapping[i] = {"match": best_i, "r2": best_r, "mapped": True if best_r >= threshold else False}
    return mapping

def map_contributions(W1, H1, W2, H2, threshold=0.6):
    mapping = {}
    matrices1 = {}
    matrices2 = {}
    for i in range(H.shape[0]):
        H_i = H1[i]
        W_i = W1[:, i]
        W_i = W_i.reshape(len(W_i), 1)
        conc = np.matmul(W_i, [H_i]).sum(axis=0)
        matrices1[i] = conc
        H2_i = H2[i]
        W2_i = W2[:, i]
        W2_i = W2_i.reshape(len(W2_i), 1)
        conc2 = np.matmul(W2_i, [H2_i]).sum(axis=0)
        matrices2[i] = conc2

    for i in range(H.shape[0]):
        m1_i = matrices1[i]
        best_i = i
        best_r = calculate_factor_correlation(m1_i, matrices2[i])
        for j in range(H.shape[0]):
            if j == i:
                pass
            j_r2 = calculate_factor_correlation(m1_i, matrices2[j])
            if j_r2 > best_r:
                best_r = j_r2
                best_i = j
        mapping[i] = {"match": best_i, "r2": best_r, "mapped": True if best_r >= threshold else False}
    return mapping

In [ ]:
rng = np.random.default_rng(seed=0)
bootstrap_datasets = {}
for i in range(bootstrap_n):
    seed_i = rng.integers(low=0, high=1e10, size=1)
    _V = cosrc.deepcopy(V)
    _U = cosrc.deepcopy(U)
    _W = cosrc.deepcopy(base_W)
    bs_data, bs_uncertainty, bs_W, bs_index = block_resample(data=_V, uncertainty=_U, W=_W, block_size=block_size, seed=seed_i, overlapping=False)
    # bs_data, bs_uncertainty, bs_W, bs_index = resample(data=_V, uncertainty=_U, W=_W, seed=seed_i)
    bootstrap_datasets[i] = {"data": bs_data, "uncertainty": bs_uncertainty, "W": bs_W, "index": bs_index}

In [ ]:
import pandas as pd

bs_i = 0
bs_index = bootstrap_datasets[0]["index"][bs_i]
print(f"bs_i: {bs_i}, bs_index: {bs_index}")
V_df = pd.DataFrame(data=V, columns=data_handler.features)
V_df.iloc[bs_index:bs_index+block_size]
# V_df.describe()

In [ ]:
df_0 = pd.DataFrame(data=bootstrap_datasets[0]['data'], columns=data_handler.features)
df_0.iloc[bs_i:bs_i+block_size]
# df_0.describe()

In [ ]:
# bootstrap_datasets[0]["index"]

In [ ]:
# BS - step 2
from esat.utils import q_loss, qr_loss

# optimized = True
base_Qtrue = nmf_models.results[nmf_models.best_model]["Q(true)"]
base_Qrobust = nmf_models.results[nmf_models.best_model]["Q(robust)"]
base_seed = nmf_models.results[nmf_models.best_model]["seed"]
rng = np.random.default_rng(seed=0)

print(f"Base Model - Q(true): {base_Qtrue}, Q(robust): {base_Qrobust}, factors: {factors}, Method: {method}, seed: {seed}")
bs_results = {}
for i in range(bootstrap_n):
    seed_i = rng.integers(low=0, high=1e10, size=1)[0]
    bs_V = bootstrap_datasets[i]["data"]
    bs_U = bootstrap_datasets[i]["uncertainty"]
    bs_W = bootstrap_datasets[i]["W"]
    bs_i_nmf = NMF(V=bs_V, U=bs_U, factors=factors, method=method, seed=seed_i, optimized=optimized, verbose=True)
    bs_i_nmf.initialize(H=base_H)
    bs_i_nmf.train(max_iter=max_iterations, converge_delta=converge_delta, converge_n=converge_n, robust_mode=False, robust_n=100, robust_alpha=4)
    bs_i_results = {"H": bs_i_nmf.H, "W": bs_i_nmf.W, "Q(true)": bs_i_nmf.Qtrue, "Q(robust)": bs_i_nmf.Qrobust}
    bs_results[i] = bs_i_results

In [ ]:
contr_mappings = map_contributions(W1=bs_results[0]["W"], H1=bs_results[0]["H"], W2=base_W, H2=base_H)
contr_mappings

In [ ]:
profile_mappings = map_factors(H1=bs_results[0]["H"], H2=base_H)
profile_mappings

In [ ]:
from esat.error.bootstrap import Bootstrap

In [ ]:
# BS parameters
data = V
uncertainty = U
model_selected = nmf_models.best_model
nmf_results = nmf_models.results[model_selected]
feature_labels = data_handler.features
method = method
optimized = True

bootstrap_n = 20
block_size = 4
threshold = 0.6
seed = seed

In [ ]:
bs = Bootstrap(data=data, uncertainty=uncertainty, model_selected=model_selected, nmf_results=nmf_results, feature_labels=feature_labels, method=method, optimized=optimized, bootstrap_n=bootstrap_n, block_size=block_size, threshold=threshold, seed=seed)

In [ ]:
bs.run()

In [ ]:
bs.summary()

In [ ]:
factor_i = 0

In [ ]:
factor_results = bs.factor_tables[factor_i]
factor_df = pd.DataFrame(factor_results, columns=data_handler.features)
factor_df[factor_df < 1e-5] = 0.0
base_factor = bs.base_H[factor_i]
base_factor[base_factor < 1e-5] = 0.0
base_df = pd.DataFrame(base_factor.reshape(1, len(base_factor)), columns=data_handler.features)

q3 = factor_df.quantile(0.75)
q1 = factor_df.quantile(0.25)
iqr = base_df.iloc[0].between(q1, q3)
results = {"features": data_handler.features, "Base Run Profile": base_factor, "Within IQR": iqr, "BS Mean": factor_df.mean(), "BS Std. Dev.": factor_df.std(), "BS 5th": factor_df.quantile(0.05), "BS 25th": factor_df.quantile(0.25), "BS Median": factor_df.median(), "BS 75th": factor_df.quantile(0.75), "BS 95th": factor_df.quantile(0.95)}
# results

In [ ]:
bs.mapping_df

In [ ]:
m_table = go.Figure(data=[go.Table(header=dict(values=bs.mapping_df.columns), cells=dict(values=bs.mapping_df.values.T))])
m_table.update_layout(width=800, height=400)
m_table.show()

In [ ]:
bs_profiles = {}
for i in range(bs.factors):
    profile = []
    for r_k, r_v in bs.bs_results.items():
        p_f = r_v["H"]
        p_fn = p_f / p_f.sum(axis=0)
        profile.append(p_fn[i])
    bs_profiles[i] = profile
f_data = np.array(bs_profiles[0])
f_data[:,0].shape

In [ ]:
factor_i = 0
base_data = bs.base_H
base_ndata = base_data / base_data.sum(axis=0)
f_data = np.array(bs_profiles[factor_i])
f_plot = go.Figure()
for i in range(len(bs.feature_labels)):
    i_data = 100 * f_data[:, i]
    f_plot.add_trace(go.Box(name=bs.feature_labels[i], y=i_data, boxpoints='outliers', notched=True, marker_color='rgb(107,174,214)', line_color='rgb(107,174,214)', marker_size=4, line_width=1))
f_plot.add_trace(go.Scatter(x=bs.feature_labels, y=100 * base_ndata[factor_i], mode='markers', marker=dict(color='red', size=4), name="Base"))
f_plot.update_layout(title=f"Variability in Percentage of Species - Model {bs.model_selected} - Factor {factor_i} ", width=1200, height=600, showlegend=False)
f_plot.update_yaxes(title_text="Percentage", range=[0, 100])
f_plot.show()

In [ ]:
base_ndata[0].shape

In [ ]:
bs_factor_contributions = {}
for i in range(bs.factors):
    contributions = []
    for r_k, r_v in bs.bs_results.items():
        i_H = [r_v["H"][i]]
        i_W = r_v["W"][:, i]
        i_W = i_W.reshape(len(i_W), 1)
        i_WH = np.matmul(i_W, i_H)
        i_sum = i_WH.sum(axis=0)
        contributions.append(i_sum)
    bs_factor_contributions[i] = contributions

In [ ]:
factor_i = 0
base_Wi = bs.base_W[:, factor_i]
base_Wi = base_Wi.reshape(len(base_Wi), 1)
base_Hi = [bs.base_H[factor_i]]
base_sums = np.matmul(base_Wi, base_Hi).sum(axis=0)
base_sums[base_sums < 1e-4] = 1e-4
c_data = np.array(bs_factor_contributions[factor_i])
c_plot = go.Figure()
for i in range(len(bs.feature_labels)):
    i_data = c_data[:, i]
    i_data[i_data < 1e-4] = 1e-4
    c_plot.add_trace(go.Box(name=bs.feature_labels[i], y=i_data, boxpoints='outliers', notched=True, marker_color='rgb(107,174,214)', line_color='rgb(107,174,214)', marker_size=4, line_width=1))
c_plot.add_trace(go.Scatter(x=bs.feature_labels, y=base_sums, mode='markers', marker=dict(color='red', size=4), name="Base"))
c_plot.update_layout(title=f"Variability in Concentration of Species - Model {bs.model_selected} - Factor {factor_i} ", width=1200, height=600, showlegend=False)
c_plot.update_yaxes(title_text="Concentration (log)", type="log")
c_plot.show()

In [ ]:
# factor_i = 0
# base_Wi = bs.base_W[:, factor_i]
# base_Wi = base_Wi.reshape(len(base_Wi), 1)
# base_Hi = [bs.base_H[factor_i]]
# base_sums = np.matmul(base_Wi, base_Hi).sum(axis=0)
# base_sums[base_sums < 1e-4] = 1e-4
# c_data = np.array(bs_factor_contributions[factor_i])
# c_data[c_data < 1e-4] = 1e-4
# e_plot = go.Figure()
# e_plot.add_trace(go.Bar(x=bs.feature_labels, y=c_data.max(axis=0) - c_data.min(axis=0), base=c_data.min(axis=0), name="Bootstrap", marker_color='rgb(158,202,225)', marker_line_color='rgb(8,48,107)'))
# e_plot.add_trace(go.Scatter(x=bs.feature_labels, y=base_sums, mode='markers', name="Base", marker=dict(size=12, color="red", symbol="line-ew", line_width=1, line_color="red")))
# e_plot.update_layout(title=f"Error Estimation Concentration Summary - Model {bs.model_selected} - Factor {factor_i}", width=1200, height=600, showlegend=True)
# e_plot.update_traces(selector=dict(type="bar"), hovertemplate='Max: %{value}<br>Min: %{base}')
# e_plot.update_yaxes(title_text="Concentration (log)", type="log")
# e_plot.show()

In [ ]:
from esat.error.error import Error

In [ ]:
error = Error(bs=bs)
error.plot_summary(factor_i=0)